In [ ]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
import json, sys, os
from IPython.display import Markdown

load_dotenv()
client = Anthropic()

model = "claude-haiku-4-5"
temperature = 0.0
max_tokens = 1024
stop_sequences = []
system_prompt = ""
tools = []

In [ ]:
sys.path.append(os.path.abspath(".."))  
from tools.live_stock_price import get_live_stock_price

In [ ]:
get_live_stock_price_schema = {
  "name": "get_live_stock_price",
  "description": "Fetch the latest stock quote for a given ticker symbol (e.g., AAPL, INFOBEAN) using FinHub API. Caches the result for 24 hours to avoid repeated API calls.",
  "input_schema": {
    "type": "object",
    "properties": {
      "stock_ticker": {
        "type": "string",
        "description": "The stock ticker symbol (e.g., 'AAPL', 'INFOBEAN', 'GOOGL'). Must be a valid symbol supported by FinHub."
      }
    },
    "required": ["stock_ticker"]
  }
}

In [ ]:
tool_registry = {
    "get_live_stock_price": get_live_stock_price,
}

In [ ]:
messages = []

def append_message(role, message):
    new_message = {"role" : role, "content": message}
    messages.append(new_message)
    return messages

def chat(message: str, temperature: float=temperature, 
        system_prompt: str = "", stop_sequences: list = [],
        tools = [get_live_stock_price_schema]):
    append_message("user", message)
    prefill = "```json"
    new_message = messages + [{"role": "assistant", "content": prefill}]
    response = client.messages.create(
        model=model, max_tokens=max_tokens, messages=new_message,
        system=system_prompt, temperature=temperature, stop_sequences=stop_sequences,
        tools = tools
    )
    # reply = response.content[0].text --> this will not work with ToolUseBlock
    append_message("assistant", response)
    return response

In [ ]:
response = chat('Get live stock price for AAPL')
print(response)

In [ ]:
# Identify if tool use block is present in response
for block in response.content:
    if(block.type == "tool_use"):
        print(f"Calling tool {block.name}")
        # Call the function with the arguments
        try:
            tool_use_id = block.id
            tool_input = block.input 
            tool_name = block.name
            func = tool_registry[tool_name]  # unpack dict as keyword args
            result = func(**tool_input)
            print(f"Function returned {result}")
            is_error = False  
        except TypeError as e:
            # raise TypeError(f"Error calling {tool_name}: {e}")
            result = str(e)
            is_error = True
        finally:
            # Send the response back to the LLM
            messages.append(
                {
                    "tool_use_id" : tool_use_id,
                    "type": "tool_result",
                    "content": result,
                    "is_error": is_error
                }
            )

print(messages)